<div dir="rtl" style="font-family: 'Noto Naskh Arabic',serif; text-align:right; line-height:1.9">

# 🏥 نموذج التنبؤ بالوصول إلى المستشفيات في اليمن
### Yemen Hospital Accessibility Prediction Model

**دفتر عملي مُعدّ للمناقشة — يُدرّب النموذج فعليًا ويحسب النتائج من الصفر.**

**إعداد وتطوير:** المهندس وهيب أمين عبد الجليل أحمد سعد  
**البرنامج:** برنامج إعداد مهندسي الذكاء الاصطناعي  
**الجهة:** مركز تطوير التعليم التقني والبحث العلمي – ذمار  
**العمل:** المشروع الختامي لدورة تعلم الآلة (Machine Learning)  
**الإشراف العلمي:** أ.د. خالد الحسيني

**الهدف:** التنبؤ بالنسبة المئوية للسكان الذين يُقدّر أنهم قادرون على الوصول إلى مستشفى خلال ساعة واحدة في المديريات اليمنية؛ اعتمادًا على مؤشر GAIA النموذجي لعام 2030.

> **تنبيه للمناقشة:** هذه *مسألة انحدار Regression* وليست تصنيفًا Classification. لذلك نستخدم MAE وRMSE وR²؛ لا نقول إن `100 - MAE` هي «الدقة».

</div>

<div dir="rtl" style="text-align:right">

## خارطة التجربة أمام اللجنة

1. فهم البيانات وفحص سلامتها، وتحديد الهدف والميزات المسموحة.
2. شرح سبب حجز **محافظات كاملة** عند التقييم، بدل توزيع المديريات عشوائيًا.
3. تدريب النموذج المرجعي **EXP_0008** على 27 ميزة و5 طيات خارجية.
4. تنفيذ التحسين **EXP_0012**: تحويل الهدف إلى Logit، ثم اختيار وزن المزج من ثلاث طيات داخلية تدريبية فقط.
5. عرض النتائج **الحقيقية المحسوبة** ومقارنتها بالنتائج التاريخية مع بيان تأثير اختلاف إصدارات المكتبات.
6. إنشاء نموذجين نهائيين من جميع بيانات التدريب للاستدلال، ثم تجربة سيناريو افتراضي مباشر.

**آخر أفضل نتيجة استكشافية مسجلة:** EXP_0012: MAE = **12.8882 نقطة مئوية**، RMSE = **18.1766**، R² = **0.72585**.  
**المرجع المستقر المحمي:** EXP_0008: MAE = **13.4509**، RMSE = **18.2778**، R² = **0.72278**.

الأرقام السابقة **توقعات للمطابقة** فقط؛ يُعاد حساب الأرقام الناتجة في الخلايا التالية. ولا تُعدّ نتيجة اختبار خارجي مستقل لأن الطيات استُخدمت مرارًا أثناء التطوير.

</div>

<div dir="rtl" style="text-align:right">

## ١ — تجهيز بيئة العمل في Kaggle

تعمل نسخة هذا الدفتر **حتى لو لم تضف بيانات Kaggle يدويًا**؛ إذ تحتوي على نسخة احتياطية مضغوطة من البيانات الثلاثة المطلوبة، مع SHA-256 للتحقق من التطابق. وإذا رفعت ملفات الحزمة كمجموعة بيانات Kaggle، يفضّل الدفتر قراءتها من `/kaggle/input`.

في Kaggle اضغط **Run All**. يجب تفعيل الإنترنت فقط إن كانت مكتبة `catboost` غير مثبتة؛ لا يحتاج التدريب إلى GPU. الزمن يختلف بحسب البيئة. احفظ تشغيل الدفتر بنهاية المناقشة.

</div>

In [ ]:
# المكتبات اللازمة للتدريب والتحليل؛ يمكن تشغيل الدفتر على CPU.
import sys, subprocess, importlib.util, os, json, time, hashlib, gzip, base64, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logit, expit
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
import matplotlib.pyplot as plt

# تثبيت CatBoost عند الحاجة فقط؛ لا نثبت أو نحدث الحزم المثبتة تلقائياً.
if importlib.util.find_spec('catboost') is None:
    print('CatBoost غير مثبتة؛ نحتاج اتصالاً بالإنترنت لتثبيتها. يمكن استبدال هذه الخطوة بتثبيت مسبق.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost==1.2.10'])
from catboost import CatBoostRegressor
import sklearn, catboost

# نضبط خيوط CatBoost من أجل إعادة إنتاج أكثر استقراراً.
SEED = 2030
THREADS = 4
print('Python:',sys.version.split()[0], '| scikit-learn:',sklearn.__version__, '| CatBoost:',catboost.__version__)

<div dir="rtl" style="text-align:right">

## ٢ — تحميل البيانات والتحقق من سلامتها

لدينا **335 مديرية** تابعة إلى **22 محافظة**. يتضمن الملف الأصلي تفاصيل أخرى قد تسبب تسربًا من الهدف، ولذلك **لا نُمرّرها مطلقًا إلى CatBoost**.

- `ADM2_PCODE`: رمز المديرية، للربط فقط.
- `ADM1_PCODE`: رمز المحافظة، لتجميع الطيات فقط.
- `hospital_access_pct`: متغير الهدف من بيانات GAIA، **لا يدخل ضمن الميزات**.
- `outer_fold`: تعيين ثابت للطية الخارجية، **لا يدخل ضمن الميزات**.
- 21 ميزة سكانية وصحية وتفاعلية، و6 ميزات من الطرق والمستشفيات المرسومة في OSM.

> ملفات GEO8 ونتائج OOF المرجعية واردة في الحزمة لأغراض التوثيق. وهذه بيانات **مُحضّرة ومدقّقة سابقًا**؛ لسنا بصدد تنزيل GeoTIFF أو إعادة بناء شبكة الطرق خلال المناقشة.

</div>

In [ ]:
# النسخة المضمنة مضغوطة ومحمية ببصمات SHA-256؛ لا تدخل نتائج OOF المرجعية في التدريب.
EMBEDDED_DATA = {
    '01_Yemen_ADM2_335_27_Features_Target_Folds.csv': (
        '...'
    ),
    '02_Source_ADM2_Raw_For_Engineering.csv': (
        '...'
    ),
    '03_Geospatial_Six_Approved_Features.csv': (
        '...'
    ),
}
EMBEDDED_SHA256 = {'01_Yemen_ADM2_335_27_Features_Target_Folds.csv': 'eeb86acb1e919a0a462b7e61e60559d78d2dadf260c805d3598bf3e2bf9bb845', '02_Source_ADM2_Raw_For_Engineering.csv': 'd0c4833963a742e1ceb327cd03670b21b28e9575451e8a77cd623e6e65486b83', '03_Geospatial_Six_Approved_Features.csv': 'b3b6ef920ed94cb035d1b1750cdd8db484ef8755ae8a44cdd9d64cac845dfabc'}

In [ ]:
# نختار الملفات المباشرة من Kaggle إن وُجدت؛ وإلا ننشئها من النسخة المضمنة.
DATA_FILES = sorted(EMBEDDED_DATA)
DATA_ROOT = None
LOOK_IN = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle/input'), Path('/mnt/data')]
for parent in LOOK_IN:
    if not parent.exists():
        continue
    paths = sorted(parent.glob('**/01_Yemen_ADM2_335_27_Features_Target_Folds.csv')) if str(parent) in ('/kaggle/input',) else list(parent.glob('data/01_Yemen_ADM2_335_27_Features_Target_Folds.csv'))
    for path in paths:
        if all((path.parent / f).is_file() for f in DATA_FILES):
            DATA_ROOT = path.parent
            break
    if DATA_ROOT is not None: break

if DATA_ROOT is None:
    DATA_ROOT = Path('/kaggle/working/yemen_health_demo_data') if Path('/kaggle/working').exists() else Path.cwd()/'yemen_health_demo_data'
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for name, chunks in EMBEDDED_DATA.items():
        (DATA_ROOT/name).write_bytes(gzip.decompress(base64.b64decode(''.join(chunks))))
    print('✅ تم تجهيز نسخة البيانات المضمنة داخل الدفتر.')
else:
    print('✅ تم العثور على البيانات ضمن المسار:',DATA_ROOT)

# نقارن المحتوى ببصمات نسختنا المعتمدة قبل حساب أي نتيجة.
for name, expected_hash in EMBEDDED_SHA256.items():
    actual_hash = hashlib.sha256((DATA_ROOT/name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f'تحذير: تغيّرت بيانات الملف: {name}'
print('✅ اجتازت ملفات البيانات الثلاثة فحص SHA-256.')

frozen = pd.read_csv(DATA_ROOT/'01_Yemen_ADM2_335_27_Features_Target_Folds.csv',
                     dtype={'ADM2_PCODE':'string','ADM1_PCODE':'string'})
raw_data = pd.read_csv(DATA_ROOT/'02_Source_ADM2_Raw_For_Engineering.csv',
                       dtype={'ADM2_PCODE':'string','ADM1_PCODE':'string'})
geo_data = pd.read_csv(DATA_ROOT/'03_Geospatial_Six_Approved_Features.csv',
                       dtype={'ADM2_PCODE':'string','ADM1_PCODE':'string'})
print(f'عدد المديريات: {len(frozen)}, عدد المحافظات: {frozen.ADM1_PCODE.nunique()}')
display(frozen[['ADM2_PCODE','ADM1_PCODE','hospital_access_pct','outer_fold']].head(7))

<div dir="rtl" style="text-align:right">

## ٣ — كيف صُنعت الميزات الـ27؟

حتى تكون التجربة مفهومة أمام اللجنة، سنعيد اشتقاق الميزات الـ21 من بيانات المشروع الأصلية، ثم نضيف **6 مؤشرات مكانية جاهزة** من حسابات OSM السابقة، ونقارن النتائج بملف الميزات المجمّد قبل التدريب. بهذه الطريقة نستطيع أن نثبت أن المدخلات ليست ناتجة عن هدف الوصول الصحي.

أمثلة: `pop_density = total_pop / area_km2`، و`hospitals_per_100k = hospitals_count / total_pop × 100000`. لا يعني ذلك أن زيادة المستشفيات ستؤدي بالضرورة إلى تحسن الوصول الفعلي بالمقدار المتنبأ به.

</div>

In [ ]:
# قائمة السماح الصريحة: لا نسمح لأي عمود آخر بالدخول إلى التدريب.
BASE_FEATURES = [
 'total_pop','hospitals_count','primary_healthcare_count','rural_pop_perc','area_km2',
 'pop_density','hospitals_per_100k','primary_healthcare_per_100k','log_total_pop',
 'RAI_total_pop','served_rural_pct','unserved_rural_pct',
 'road_x_hospitals_density','road_x_healthcare_density','hospital_present',
 'log_pop_density','healthcare_facilities_per_km2','hospitals_per_km2',
 'facility_density_x_rural','facility_density_x_road','hospital_presence_x_rural_x_road'
]
GEO_FEATURES = [
 'osm_motor_road_km_per_km2','osm_motor_road_km_per_10k_pop',
 'osm_major_road_share','osm_track_road_share',
 'osm_mapped_hospitals_25km_centroid','osm_mapped_hospitals_50km_centroid'
]
FEATURES = BASE_FEATURES + GEO_FEATURES
FORBIDDEN = {'ADM2_PCODE','ADM1_PCODE','hospital_access_pct','outer_fold','access_pop_hospitals_1h'}
assert len(FEATURES)==27 and not FORBIDDEN.intersection(FEATURES)

def engineer_approved_features(source):
    """اشتقاق ميزات المديرية من خصائصها المسموحة، دون استخدام أي قيم للهدف."""
    f = source.copy()
    for c in ['total_pop','hospitals_count','primary_healthcare_count',
              'rural_pop_perc','RAI_total_pop','area_km2']:
        f[c] = pd.to_numeric(f[c],errors='coerce')
    pop = f['total_pop'].where(f['total_pop']>0)
    area = f['area_km2'].where(f['area_km2']>0)
    hospitals = f['hospitals_count']
    phc = f['primary_healthcare_count']
    rural = f['rural_pop_perc']
    rai = f['RAI_total_pop'].where(f['RAI_total_pop'].between(0,100))
    f['log_total_pop'] = np.log1p(pop)
    f['pop_density'] = pop/area
    f['hospitals_per_100k'] = 1e5*hospitals/pop
    f['primary_healthcare_per_100k'] = 1e5*phc/pop
    f['served_rural_pct'] = rural*rai/100
    f['unserved_rural_pct'] = rural*(1-rai/100)
    f['road_x_hospitals_density'] = f['hospitals_per_100k']*rai/100
    f['road_x_healthcare_density'] = f['primary_healthcare_per_100k']*rai/100
    f['hospital_present'] = (hospitals>0).astype(float)
    f['log_pop_density'] = np.log1p(f['pop_density'].where(f['pop_density']>=0))
    f['healthcare_facilities_per_km2'] = (hospitals+phc)/area
    f['hospitals_per_km2'] = hospitals/area
    log_facility_density = np.log1p(f['healthcare_facilities_per_km2'].where(lambda s:s>=0))
    f['facility_density_x_rural'] = log_facility_density*(rural/100)
    f['facility_density_x_road'] = log_facility_density*(rai/100)
    f['hospital_presence_x_rural_x_road'] = f['hospital_present']*(rural/100)*(rai/100)
    for c in BASE_FEATURES:
        f[c] = pd.to_numeric(f[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    return f

joined = raw_data.merge(geo_data[['ADM2_PCODE','ADM1_PCODE']+GEO_FEATURES],
                        on=['ADM2_PCODE','ADM1_PCODE'],validate='one_to_one',sort=False)
features_rebuilt = engineer_approved_features(joined)
reconstructed = features_rebuilt[['ADM2_PCODE','ADM1_PCODE']+BASE_FEATURES+GEO_FEATURES]
compared = frozen[['ADM2_PCODE','ADM1_PCODE']+FEATURES].merge(
    reconstructed,on=['ADM2_PCODE','ADM1_PCODE'],suffixes=('_frozen','_rebuilt'),validate='one_to_one')
max_feature_diff = max(float(np.max(np.abs(compared[f+'_frozen']-compared[f+'_rebuilt']))) for f in FEATURES)
assert max_feature_diff < 1e-7, f'فشل تطابق الميزات: {max_feature_diff}'
assert len(frozen)==335 and frozen.ADM2_PCODE.is_unique and frozen.ADM1_PCODE.nunique()==22
assert np.isfinite(frozen[FEATURES].to_numpy(float)).all()
assert frozen.hospital_access_pct.between(0,100).all()
assert sorted(frozen.outer_fold.unique().tolist()) == [1,2,3,4,5]
for fold in sorted(frozen.outer_fold.unique()):
    gov_train=set(frozen.loc[frozen.outer_fold!=fold,'ADM1_PCODE'])
    gov_test=set(frozen.loc[frozen.outer_fold==fold,'ADM1_PCODE'])
    assert not gov_train.intersection(gov_test), 'تسرب المحافظات عبر الطيات!'
print(f'✅ تم إعادة اشتقاق 27 ميزة بنجاح؛ أقصى فرق عددي = {max_feature_diff:.2e}')
print('✅ البيانات سليمة: 335 مديرية، 22 محافظة، 5 طيات لا تتداخل فيها المحافظات.')

<div dir="rtl" style="text-align:right">

## ٤ — لماذا نقسم حسب المحافظات؟

إذا وضعنا بعض مديريات المحافظة نفسها في التدريب وبعضها في الاختبار، فربما يستفيد النموذج من تشابهها الجغرافي والاجتماعي، ويعطينا تقييمًا متفائلًا. لذلك نفصل **محافظات كاملة** بخمس طيات خارجية محفوظة سابقًا.

- **Outer folds:** قياس التعميم على محافظات محجوبة أثناء التدريب.
- **Inner GroupKFold:** اختيار وزن المزج دون لمس أهداف المحافظات المحجوبة.
- **OOF prediction:** كل نتيجة تقييم تصدر لنقطة لم تكن ضمن تدريب النموذج الذي تنبّأ بها.

> هذه النتيجة استكشافية؛ فقد اطّلعنا على نتائج هذه الطيات في تجارب سابقة. لا نسميها قياسًا مستقلاً نهائيًا.

</div>

In [ ]:
# نظهر المحافظات المستبعدة من التدريب في كل جولة.
fold_summary = []
for fold in range(1,6):
    test_rows = frozen[frozen.outer_fold==fold]
    fold_summary.append({'الطية':fold,'عدد المديريات':len(test_rows),
                         'المحافظات المحجوبة':', '.join(sorted(test_rows.ADM1_PCODE.unique()))})
display(pd.DataFrame(fold_summary))
X = frozen[FEATURES].copy()
y = frozen['hospital_access_pct'].to_numpy(dtype=float)
groups = frozen['ADM1_PCODE'].astype(str).to_numpy()
outer_fold = frozen['outer_fold'].to_numpy(dtype=int)

def new_catboost():
    """النموذج نفسه المعتمد سابقاً دون ضبط جديد على الطيات الخارجية."""
    return CatBoostRegressor(loss_function='MAE',iterations=300,depth=4,
                             learning_rate=0.05,l2_leaf_reg=5.0,
                             random_seed=2030,verbose=False,thread_count=THREADS,
                             allow_writing_files=False)

def compute_metrics(y_true,pred):
    return {'MAE_pp':float(mean_absolute_error(y_true,pred)),
            'RMSE_pp':float(np.sqrt(mean_squared_error(y_true,pred))),
            'R2':float(r2_score(y_true,pred))}

<div dir="rtl" style="text-align:right">

## ٥ — تدريب النموذج المرجعي EXP_0008 من الصفر

**CatBoost** نموذج أشجار تعزيزية (Gradient Boosting) مناسب للعلاقات غير الخطية. اخترنا خسارة `MAE` لأن الهدف نسبة وصول، ونريد تقليل متوسط الخطأ بالنقاط المئوية. الإعدادات مثبتة: `iterations=300`، `depth=4`، `learning_rate=0.05`، `l2_leaf_reg=5`، `seed=2030`.

هذه الخلية **تدرّب خمسة نماذج بالفعل**، كل واحد على أربع مجموعات محافظات ويتنبأ على المجموعة الخامسة. لا تقرأ التنبؤات المرجعية لإنشاء النتيجة.

</div>

In [ ]:
start_time = time.perf_counter()
baseline_oof = np.full(len(y),np.nan)
for fold in range(1,6):
    tr = np.flatnonzero(outer_fold != fold)
    te = np.flatnonzero(outer_fold == fold)
    assert not set(groups[tr]).intersection(groups[te])
    imputer = SimpleImputer(strategy='median')
    x_tr = imputer.fit_transform(X.iloc[tr])  # نتعلم التعويض من التدريب فقط.
    x_te = imputer.transform(X.iloc[te])      # ثم نطبّق التعويض على الاختبار.
    mdl = new_catboost()
    mdl.fit(x_tr,y[tr])
    baseline_oof[te] = np.clip(mdl.predict(x_te),0,100)
    print(f'الطية {fold}/5 | مديريات الاختبار: {len(te):3} | MAE: {mean_absolute_error(y[te],baseline_oof[te]):.4f} نقطة')
assert np.isfinite(baseline_oof).all()
BASE_METRICS = compute_metrics(y,baseline_oof)
print('\n✅ تمت إعادة تدريب المرجع وقياسه من الصفر:',BASE_METRICS)
print('زمن هذه المرحلة: %.2f ثانية' % (time.perf_counter()-start_time))

<div dir="rtl" style="text-align:right">

## ٦ — لماذا كانت EXP_0012 أفضل استكشافيًا؟

نحوّل النسبة `p = y / 100` إلى لوغيت:  
\[z = \log\left(\frac{p}{1-p}ight)\]

ثم ندرّب CatBoost على `z` ونعيد تنبؤه إلى نسبة مئوية باستخدام `sigmoid`. قبل اللوغيت نحصر `p` بين `0.0015` و`0.9985` كي لا يحدث لوغاريتم صفر.

**المزج** بين توقع CatBoost الأصلي وتوقع Logit:
\[\widehat{y}_{blend}=(1-\alpha)\widehat{y}_{base}+\alpha\widehat{y}_{logit}\]

**السؤال المتوقع من المناقش:** لماذا لا نختار α من نتائج الاختبار الخارجي؟  
**الإجابة:** لأن ذلك يُسرّب معلومات عن المحافظات المحجوبة ويجعل التقييم متفائلًا؛ لذا نختاره بثلاث طيات داخلية خاصة ببيانات التدريب لكل جولة خارجية فقط.

</div>

In [ ]:
# تجربة التحسين الحقيقية: 5 طيات خارجية × 3 طيات داخلية.
EPS = 0.0015
ALPHA_GRID = [0.0,0.10,0.20,0.25,0.35,0.50,0.65,0.75,1.0]
logit_oof = np.full(len(y),np.nan)
blend_oof = np.full(len(y),np.nan)
alpha_per_row = np.full(len(y),np.nan)
fold_details = []
start_time = time.perf_counter()
for fold in range(1,6):
    tr=np.flatnonzero(outer_fold!=fold)
    te=np.flatnonzero(outer_fold==fold)
    inner=GroupKFold(n_splits=3)
    inner_base=np.full(len(tr),np.nan)
    inner_logit=np.full(len(tr),np.nan)
    # التدريب الداخلي: لا نستخدم أي صف من المحافظات الخارجية المحجوبة.
    for itr, iva in inner.split(X.iloc[tr],y[tr],groups[tr]):
        fit_idx=tr[itr]
        val_idx=tr[iva]
        assert not set(groups[fit_idx]).intersection(groups[val_idx])
        imputer=SimpleImputer(strategy='median')
        xf=imputer.fit_transform(X.iloc[fit_idx]); xv=imputer.transform(X.iloc[val_idx])
        base_model=new_catboost(); base_model.fit(xf,y[fit_idx])
        inner_base[iva]=np.clip(base_model.predict(xv),0,100)
        logit_model=new_catboost()
        transformed_target=logit(np.clip(y[fit_idx]/100.0,EPS,1-EPS))
        logit_model.fit(xf,transformed_target)
        inner_logit[iva]=np.clip(100*expit(logit_model.predict(xv)),0,100)
    assert np.isfinite(inner_base).all() and np.isfinite(inner_logit).all()
    # اختيار وزن المزج: نفحص الأهداف الداخلية التدريبية فقط.
    scores={a:mean_absolute_error(y[tr],(1-a)*inner_base+a*inner_logit) for a in ALPHA_GRID}
    alpha=min(scores,key=scores.get)
    # إعادة التدريب على جميع المديريات التدريبية للطية الخارجية.
    imputer=SimpleImputer(strategy='median')
    xf=imputer.fit_transform(X.iloc[tr]); xv=imputer.transform(X.iloc[te])
    trained_logit=new_catboost()
    trained_logit.fit(xf,logit(np.clip(y[tr]/100.0,EPS,1-EPS)))
    logit_oof[te]=np.clip(100*expit(trained_logit.predict(xv)),0,100)
    # المرجع baseline_oof[te] تم إنتاجه من تدريب تلك الطية نفسها في المرحلة السابقة.
    blend_oof[te]=np.clip((1-alpha)*baseline_oof[te]+alpha*logit_oof[te],0,100)
    alpha_per_row[te]=alpha
    fold_details.append({'الطية':fold,'وزن Logit المختار':alpha,
                         'MAE المرجع':mean_absolute_error(y[te],baseline_oof[te]),
                         'MAE التحسين':mean_absolute_error(y[te],blend_oof[te])})
    print(f'الطية {fold}/5 | α={alpha:.2f} | MAE المرجع={fold_details[-1]["MAE المرجع"]:.4f} | MAE التحسين={fold_details[-1]["MAE التحسين"]:.4f}')
assert np.isfinite(logit_oof).all() and np.isfinite(blend_oof).all()
ENHANCED_METRICS=compute_metrics(y,blend_oof)
print('\n✅ هذه المقاييس مُحتسبة من التدريب الحقيقي:',ENHANCED_METRICS)
print('زمن مرحلة التحسين: %.2f ثانية' % (time.perf_counter()-start_time))
display(pd.DataFrame(fold_details))

<div dir="rtl" style="text-align:right">

## ٧ — إعلان نتائج التدريب ومقارنتها بأرقام التجربة السابقة

**MAE** هو متوسط الخطأ المطلق بين النسبة الحقيقية *وفق هدف GAIA النموذجي* والنسبة المتنبأ بها، ويُقاس هنا **بالنقاط المئوية**.  
**RMSE** يعطي وزنًا أكبر للأخطاء الكبيرة.  
**R²** هو معامل تحديد يصف نسبة التباين المفسّر وفق صياغته الإحصائية؛ ليس «دقة تصنيف» ولا نسبة مديريات صحيحة.

إذا اختلفت الأرقام قليلًا بسبب إصدار CatBoost أو المعالجات العددية، نعرض **الأرقام التي أعاد الدفتر حسابها فعلًا**، ونوضح الفارق ولا ندّعي تطابقًا غير متحقق.

</div>

In [ ]:
# نعرض النتيجة المحسوبة أولاً ثم المقارنة المرجعية للتوثيق فقط.
metrics_table=pd.DataFrame([
 {'النموذج':'EXP_0008 — مُعاد تدريبه','MAE ↓':BASE_METRICS['MAE_pp'],
  'RMSE ↓':BASE_METRICS['RMSE_pp'],'R² ↑':BASE_METRICS['R2']},
 {'النموذج':'EXP_0012 — مُعاد تدريبه','MAE ↓':ENHANCED_METRICS['MAE_pp'],
  'RMSE ↓':ENHANCED_METRICS['RMSE_pp'],'R² ↑':ENHANCED_METRICS['R2']},
 {'النموذج':'EXP_0008 — سجل التجربة السابق','MAE ↓':13.45094577,'RMSE ↓':18.27779560,'R² ↑':0.7227846754},
 {'النموذج':'EXP_0012 — سجل التجربة السابق','MAE ↓':12.88823384,'RMSE ↓':18.17657202,'R² ↑':0.7258466451},
])
display(metrics_table.style.format({'MAE ↓':'{:.4f}','RMSE ↓':'{:.4f}','R² ↑':'{:.5f}'}))
print('فرق MAE لإعادة EXP_0012 مقارنة بسجلها: %.6f نقطة مئوية' % (ENHANCED_METRICS['MAE_pp']-12.88823384))
print('فرق R² لإعادة EXP_0012 مقارنة بسجلها: %.7f' % (ENHANCED_METRICS['R2']-0.7258466451))
if ENHANCED_METRICS['MAE_pp'] < BASE_METRICS['MAE_pp']:
    print('✅ تحسن MAE في هذا التشغيل الفعلي مقارنة بالنموذج المرجعي.')
else:
    print('ℹ️ لم يتحسن MAE في هذا التشغيل؛ نعرض النتيجة كما هي بلا تعديل.')

<div dir="rtl" style="text-align:right">

## ٨ — الرسوم التحليلية وأسئلة متوقعة

**لماذا نُظهر رسومًا؟** لكي تميّز لجنة المناقشة بين التنبؤات القريبة والأخطاء الشديدة، وترى فروق المحافظات بدل الاعتماد على المتوسط الوطني فقط.

</div>

In [ ]:
# مخطط التنبؤ مقابل هدف GAIA والمقارنة بين النموذجين.
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
ax[0].scatter(y,baseline_oof,s=16,alpha=.62,label='EXP_0008')
ax[0].scatter(y,blend_oof,s=16,alpha=.62,label='EXP_0012')
ax[0].plot([0,100],[0,100],ls='--',c='black',lw=1)
ax[0].set(xlabel='GAIA target (%)',ylabel='OOF model prediction (%)',title='Observed modeled target vs predictions')
ax[0].legend()
ax[1].hist(np.abs(y-baseline_oof),bins=16,alpha=.6,label='EXP_0008')
ax[1].hist(np.abs(y-blend_oof),bins=16,alpha=.6,label='EXP_0012')
ax[1].set(xlabel='Absolute error (percentage points)',ylabel='Districts',title='Error distribution')
ax[1].legend()
plt.tight_layout();plt.show()

# عدّ الأخطاء الكبيرة في المديريات.
for k,p in [('EXP_0008',baseline_oof),('EXP_0012',blend_oof)]:
    ae=np.abs(y-p)
    print(k,'| أخطاء ≥30:',int((ae>=30).sum()),'| ≥40:',int((ae>=40).sum()),'| ≥50:',int((ae>=50).sum()))

In [ ]:
# متوسط الخطأ في كل محافظة؛ هل التحسن موزّع جغرافيًا؟
gov_results=pd.DataFrame({'ADM1_PCODE':groups,'abs_base':np.abs(y-baseline_oof),
                          'abs_blend':np.abs(y-blend_oof)})
gov_results=gov_results.groupby('ADM1_PCODE',as_index=False).agg(
    MAE_EXP_0008=('abs_base','mean'),MAE_EXP_0012=('abs_blend','mean'))
gov_results['التحسن بالنقاط']=gov_results.MAE_EXP_0008-gov_results.MAE_EXP_0012
print('محافظات تحسّنت:',int((gov_results['التحسن بالنقاط']>0).sum()),
      '| محافظات تراجعت:',int((gov_results['التحسن بالنقاط']<0).sum()))
display(gov_results.sort_values('التحسن بالنقاط',ascending=False))

<div dir="rtl" style="text-align:right">

## ٩ — تدريب النموذج النهائي للاستدلال الحي

ما سبق كان **تقييمًا خارج الطيات**. الآن، وبعد أن انتهينا من حساب المقاييس، نستطيع تدريب نموذج أساسي وآخر Logit على المديريات كلها لاستخدامهما في **اختبار مدخلات جديدة**. لا نحسب دقة هذا النموذج على بيانات تدريبه؛ لأن ذلك ليس اختبارًا صالحًا للتعميم.

نختار وزن المزج النهائي عبر GroupKFold داخلي على جميع المحافظات، ثم نعيد تدريب النموذجين على كل المديريات. هذا لا يغيّر نتائج OOF أعلاه.

</div>

In [ ]:
# اختيار وزن مزج ثابت للتشغيل الحي، ثم التدريب الكامل.
cv3=GroupKFold(n_splits=3)
all_base_cv=np.full(len(y),np.nan)
all_logit_cv=np.full(len(y),np.nan)
for tr,va in cv3.split(X,y,groups):
    imp=SimpleImputer(strategy='median')
    xf=imp.fit_transform(X.iloc[tr]); xv=imp.transform(X.iloc[va])
    m0=new_catboost();m0.fit(xf,y[tr]);all_base_cv[va]=np.clip(m0.predict(xv),0,100)
    m1=new_catboost();m1.fit(xf,logit(np.clip(y[tr]/100,EPS,1-EPS)))
    all_logit_cv[va]=np.clip(100*expit(m1.predict(xv)),0,100)
full_cv_scores={a:mean_absolute_error(y,(1-a)*all_base_cv+a*all_logit_cv) for a in ALPHA_GRID}
FINAL_ALPHA=min(full_cv_scores,key=full_cv_scores.get)
full_imputer=SimpleImputer(strategy='median')
full_X=full_imputer.fit_transform(X)
full_base=new_catboost();full_base.fit(full_X,y)
full_logit=new_catboost();full_logit.fit(full_X,logit(np.clip(y/100,EPS,1-EPS)))

# مجلد المخرجات: Kaggle/working عند التشغيل على المنصة.
OUTPUT_DIR=Path('/kaggle/working/yemen_health_defense_outputs') if Path('/kaggle/working').exists() else Path.cwd()/'yemen_health_defense_outputs'
OUTPUT_DIR.mkdir(exist_ok=True,parents=True)
full_base.save_model(str(OUTPUT_DIR/'model_EXP_0008_full.cbm'))
full_logit.save_model(str(OUTPUT_DIR/'model_EXP_0012_logit_full.cbm'))
(OUTPUT_DIR/'inference_settings.json').write_text(json.dumps({
    'feature_names':FEATURES,'epsilon':EPS,'final_blend_weight_logit':FINAL_ALPHA,
    'imputer_medians':full_imputer.statistics_.tolist(),
    'note':'Full-fit deployment models; not an independent performance evaluation.'
},ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ تم حفظ النموذجين بعد تدريب كامل على بيانات المشروع.')
print('وزن Logit النهائي المحسوب من طيات التدريب:',FINAL_ALPHA)

<div dir="rtl" style="text-align:right">

## ١٠ — اختبار نموذج حقيقي بمدخلات جديدة أمام اللجنة

اختر مديرية تمثل **نقطة انطلاق للسيناريو**، وعدّل عدد المستشفيات أو السكان. ستُعاد هندسة الميزات السكانية والصحية المتأثرة تلقائيًا، ثم يُستدعى النموذجان المحفوظان ويُحسب ناتج المزج.

> ملاحظة علمية: ميزات الطرق والمستشفيات الموجودة في نطاق 25/50 كم تظل ثابتة في سيناريو التعديل السريع. لذلك تُستخدم النتيجة لتوضيح **كيفية عمل الاستدلال** فقط، وليس لإثبات أثر بناء مستشفى على الوصول الواقعي أو التوصية بقرار تخطيطي.

</div>

In [ ]:
# مثال حي قابل للتعديل أمام اللجنة: غيّر DISTRICT_CODE أو HOSPITAL_CHANGE ثم شغّل الخلية.
DISTRICT_CODE=str(frozen.loc[0,'ADM2_PCODE'])
HOSPITAL_CHANGE=3  # عدد المستشفيات الافتراضية الإضافية؛ غيّره وقت المناقشة.
POPULATION_CHANGE_PCT=0  # مثال: 10 تعني زيادة افتراضية في السكان قدرها 10%.

def predict_live_scenario(code, additional_hospitals=0, population_change_pct=0):
    row=raw_data.loc[raw_data.ADM2_PCODE==code].copy()
    assert len(row)==1,'رمز المديرية غير موجود'
    geo_row=geo_data.loc[geo_data.ADM2_PCODE==code, ['ADM2_PCODE','ADM1_PCODE']+GEO_FEATURES]
    row['hospitals_count']=(row.hospitals_count.astype(float)+additional_hospitals).clip(lower=0)
    row['total_pop']=row.total_pop.astype(float)*(1+population_change_pct/100)
    row=row.merge(geo_row,on=['ADM2_PCODE','ADM1_PCODE'],validate='one_to_one')
    engineered=engineer_approved_features(row)
    input_values=engineered[FEATURES].to_numpy(dtype=float)
    # نستخدم نفس وسيط التعويض الذي تعلمناه من التدريب الكامل.
    input_values=np.where(np.isfinite(input_values),input_values,full_imputer.statistics_)
    p_base=np.clip(full_base.predict(input_values),0,100)
    p_logit=np.clip(100*expit(full_logit.predict(input_values)),0,100)
    result=float(np.clip((1-FINAL_ALPHA)*p_base+FINAL_ALPHA*p_logit,0,100)[0])
    return result

before=predict_live_scenario(DISTRICT_CODE,0,0)
after=predict_live_scenario(DISTRICT_CODE,HOSPITAL_CHANGE,POPULATION_CHANGE_PCT)
print('رمز المديرية:',DISTRICT_CODE)
print(f'تنبؤ النموذج الأساسي قبل التعديل: {before:.2f}%')
print(f'تنبؤ النموذج بعد السيناريو الافتراضي: {after:.2f}%')
print('التغيير في التنبؤ: %+0.2f نقطة مئوية' % (after-before))
print('هذا ناتج استدلال حسابي حي، وليس إثباتًا لنتيجة تدخل صحي حقيقي.')

<div dir="rtl" style="text-align:right">

## ١١ — حفظ ملفات التنبؤ والمقاييس للعرض والتوثيق

سنُخرج ملف OOF كاملًا؛ كل صف يضم الهدف النموذجي، وتنبؤ المرجع، وتنبؤ التحسين، ورمز المديرية والطية. **التنبؤات المنشأة هنا من التدريب، وليست بيانات مرفوعة كما هي**. نحتفظ أيضًا بملف `metrics_recomputed.json` لكي نستطيع شرح الأرقام للمناقش.

</div>

In [ ]:
# تصدير نتائج التجربة الفعلية إلى Kaggle Output.
prediction_table=frozen[['ADM2_PCODE','ADM1_PCODE','hospital_access_pct','outer_fold']].copy()
prediction_table['EXP_0008_OOF_retrained']=baseline_oof
prediction_table['EXP_0012_OOF_retrained']=blend_oof
prediction_table['EXP_0012_selected_alpha']=alpha_per_row
prediction_table['EXP_0012_absolute_error_pp']=np.abs(y-blend_oof)
prediction_table.to_csv(OUTPUT_DIR/'district_OOF_predictions_retrained.csv',index=False)
gov_results.to_csv(OUTPUT_DIR/'governorate_comparison.csv',index=False)
metrics_export={
 'project':'Yemen Hospital Accessibility Prediction Model',
 'status':'EXPLORATORY_NOT_INDEPENDENT',
 'n_districts':int(len(y)), 'n_governorates':int(len(set(groups))),
 'n_features':len(FEATURES),
 'EXP_0008_retrained':BASE_METRICS,
 'EXP_0012_retrained':ENHANCED_METRICS,
 'reference_historical_EXP_0008':{'MAE_pp':13.45094577,'RMSE_pp':18.27779560,'R2':0.7227846754},
 'reference_historical_EXP_0012':{'MAE_pp':12.88823384,'RMSE_pp':18.17657202,'R2':0.7258466451},
 'selected_inner_alphas':{str(r['الطية']):r['وزن Logit المختار'] for r in fold_details},
 'final_deployment_blend_weight':FINAL_ALPHA,
 'caveat':'Target is GAIA modeled 2030 access; folds reused during development, not an independent final test.'
}
(OUTPUT_DIR/'metrics_recomputed.json').write_text(json.dumps(metrics_export,ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ ملفات الإخراج التي أنشأها التدريب:')
for p in sorted(OUTPUT_DIR.iterdir()):print('-',p.name, f'({p.stat().st_size:,} bytes)')

<div dir="rtl" style="text-align:right">

## ١٢ — أسئلة متوقعة من لجنة المناقشة

**س1: هل هذا نموذج تصنيف؟**  
لا؛ إنه **انحدار (Regression)** يتنبأ بنسبة مستمرة بين 0 و100%.

**س2: هل R² = 0.72585 يعني أن الدقة 72.585%؟**  
لا؛ هذا **معامل تحديد**، وليس نسبة حالات صحيحة.

**س3: هل الـ MAE = 12.8882 يعني خطأ 12.8882% نسبيًا؟**  
لا؛ هو **12.8882 نقطة مئوية** في متوسط الفرق بين النسبة المستهدفة والمتنبأ بها.

**س4: هل هذه نتائج تجريبية فعلية؟**  
نعم، عند تشغيل الدفتر تُدرّب النماذج وتحسب النتائج مجددًا؛ لكن الطيات نفسها سبق فحصها أثناء تطوير المشروع، فلا نسميها اختبارًا خارجيًا جديدًا مستقلًا.

**س5: من أين جاء متغير الهدف؟**  
من **تقديرات GAIA النموذجية لإمكانية الوصول إلى المستشفيات خلال ساعة واحدة لعام 2030**، وليس من قياسات فعلية لوقت سفر مرضى عام 2030.

**س6: لماذا لا نستخدم أعمدة `access_*`؟**  
لأنها تُمثل مؤشرات الوصول التي قد تكشف الهدف مباشرة أو بصورة غير مباشرة؛ لذلك نستبعدها لتقليل تسرب الهدف.

**س7: لماذا لا يُثبت سيناريو زيادة المستشفيات أثرًا حقيقيًا؟**  
لأنه يغير مدخلات النموذج دون إعادة بناء الطرق أو مواقع المستشفيات أو نمذجة حركة السكان؛ لذا هو **استدلال افتراضي** لا دراسة سببية.

**س8: هل البيانات الجغرافية والهدف مستقلان بالكامل؟**  
ليس مؤكّدًا؛ ربما يوجد تداخل بين جرد مرافق OSM وبعض المصادر المستخدمة لإعداد GAIA، كما يوجد فرق زمني بين تحديثات OSM والنموذج المستهدف.

**س9: ماذا فعلت أثناء التدريب لمنع التسرب بين المحافظات؟**  
حافظت على طيات خارجية ثابتة، وفصلت المحافظات تمامًا بين التدريب والاختبار، واخترت وزن المزج داخل طيات تدريبية أصغر.

**س10: لماذا تدربنا نموذجًا كاملًا بعد التقييم؟**  
للاستدلال على مدخلات جديدة والاستفادة من جميع الصفوف المتاحة. **لا نستخدم خطأ التدريب الكامل دليلاً على جودة التعميم**.

### خلاصة

هذا دفتر قابل للتنفيذ في جلسة المناقشة، ونتائجه المحسوبة فعليًا هي المرجع أثناء العرض؛ بينما الأرقام التاريخية المحفوظة تبقى أساسًا للمقارنة ومراجعة قابلية إعادة الإنتاج.

</div>